# Random forest regressor tutorial

We fit `mlpackage.supervised_learning.RandomForestRegressor` on the **Diabetes** dataset: many **variance-reduction** trees on bootstrap samples, **mean** aggregation at prediction time. **`numpy.random.seed`** before **`fit`** keeps bagging reproducible.

## Prerequisites and goals

**You will**
- Train an ensemble with **`n_estimators`**, **`max_depth`**, **`max_features`**.
- Report **\(R^2\)** and **RMSE** on train and test (this class has no `score` method).
- Inspect residuals and a **truth vs prediction** scatter on held-out data.
- Compare **`n_estimators`** on the same split.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from mlpackage.supervised_learning import RandomForestRegressor

## Step 1: Load Diabetes

In [ ]:
diabetes = load_diabetes()
X = np.asarray(diabetes.data, dtype=float)
y = np.asarray(diabetes.target, dtype=float)

feature_names = list(diabetes.feature_names)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Target mean:", round(float(y.mean()), 2))

## Step 2: Train/test split

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])

## Step 3: Fit `RandomForestRegressor`

Trees partition raw feature scales here (same as the single-tree regressor tutorial). **`np.random.seed`** controls bootstrap and feature subsampling.

In [ ]:
RNG = 42
N_TREES = 80
MAX_DEPTH = 8

np.random.seed(RNG)
rf = RandomForestRegressor(
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
    max_features="sqrt",
)
rf.fit(X_train, y_train)
print("Fitted RandomForestRegressor with", N_TREES, "trees.")

## Step 4: Predict on train and test

In [ ]:
y_hat_train = rf.predict(X_train)
y_hat_test = rf.predict(X_test)

print("First 8 test predictions:", np.round(y_hat_test[:8], 2))
print("First 8 test targets:   ", np.round(y_test[:8], 2))

## Step 5: \(R^2\) and RMSE

Same definitions as in the decision tree regressor README: \(R^2 = 1 - \mathrm{SS}_{\mathrm{res}}/\mathrm{SS}_{\mathrm{tot}}\), RMSE \(= \sqrt{\mathrm{mean}((y-\hat{y})^2)}\).

In [ ]:
def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    if ss_tot == 0.0:
        return 1.0
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    return 1.0 - ss_res / ss_tot


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


print(f"Train R^2 : {r2_score(y_train, y_hat_train):.4f}")
print(f"Test R^2  : {r2_score(y_test, y_hat_test):.4f}")
print(f"Train RMSE: {rmse(y_train, y_hat_train):.4f}")
print(f"Test RMSE : {rmse(y_test, y_hat_test):.4f}")

## Step 6: Residual preview (test fold)

In [ ]:
res = y_test - y_hat_test
prev = pd.DataFrame(
    {"y_true": y_test, "y_pred": y_hat_test, "residual": res}
).head(12)
display(prev.round(3))

## Step 7: Truth vs predicted (held-out test)

Points near the diagonal indicate agreement; forest averaging often smooths extreme single-tree predictions.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_hat_test, alpha=0.75, edgecolors="black", linewidths=0.3)

lims = float(min(y_test.min(), y_hat_test.min())), float(max(y_test.max(), y_hat_test.max()))
margin = (lims[1] - lims[0]) * 0.05 if lims[1] > lims[0] else 1.0
lo, hi = lims[0] - margin, lims[1] + margin
plt.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1, label="y = ŷ")

plt.xlabel("Actual progression (held-out)")
plt.ylabel("Predicted progression (held-out)")
plt.title("Random forest regressor — test truth vs prediction")
plt.legend()
plt.axis("square")
plt.tight_layout()

_nb = Path("random_forest_regressor_tutorial.ipynb")
_out = (
    _nb.with_name("diabetes_random_forest_y_vs_yhat_scatter.png")
    if _nb.is_file()
    else Path(
        "examples/supervised_learning/random_forest_regressor/diabetes_random_forest_y_vs_yhat_scatter.png"
    )
)
_out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_out.resolve()}")
plt.show()

## Step 8: Compare `n_estimators`

Reset **`np.random.seed(RNG)`** before each refit.

In [ ]:
sizes = [20, 40, 80, 120]
rows = []

for n in sizes:
    np.random.seed(RNG)
    model = RandomForestRegressor(
        n_estimators=n,
        max_depth=MAX_DEPTH,
        max_features="sqrt",
    )
    model.fit(X_train, y_train)
    y_tr = model.predict(X_train)
    y_te = model.predict(X_test)
    rows.append(
        {
            "n_estimators": n,
            "train_r2": r2_score(y_train, y_tr),
            "test_r2": r2_score(y_test, y_te),
            "test_rmse": rmse(y_test, y_te),
        }
    )

display(pd.DataFrame(rows).round(4))